In [1]:
import os
os.chdir("..")

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from optimizer import grid_search, strategy_compare

In [3]:
SEARCH_SPACE = {
    "cooldown_long": [30, 60, 90, 180],
    "cooldown_short": [7, 14, 30, 60],
}

In [4]:
df_prices = pd.read_csv("data/test_crypto.csv", index_col=0, parse_dates=True)
df_prices

,BTC,ETH,SOL,XRP,ADA,DOT,LINK,XLM,XMR
Date,,,,,,,,,
2014-09-17,457.334015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-18,424.440002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-19,394.795990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-20,408.903992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-21,398.821014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-10-19,81853.539062,2404.815186,104.926231,1.302713,0.217995,1.153126,12.043903,0.177571,605.426392
2026-10-20,82941.000000,2446.730713,106.422340,1.334452,0.223782,1.149771,12.320927,0.181960,600.929504
2026-10-21,84011.968750,2470.766113,106.793655,1.363619,0.224979,1.141323,12.484719,0.183488,622.434082


# SEARCH

In [5]:
history_search = grid_search(
    prices=df_prices, 
    param_grid=SEARCH_SPACE, 
    n_boot=100,
    sample_size=10,
    sequence_length=365 * 4,
    )

100%|██████████| 16/16 [11:59<00:00, 44.96s/it]


In [6]:
history_search

,strategy,Target,Cash_spent,Realized_cash,Portfolio,Invested,Total_value,Bull_history,Avg_price,Num_take_profits,MDD,MA200_survival_days,RoAC,iteration
0,cooldown_long=30_cooldown_short=7,BTC,2069,3099,565,869,3665,75,31068.099326,7,-374,0.000000,85,1
1,cooldown_long=30_cooldown_short=7,LINK,2574,4781,892,400,5674,43,10.892359,3,-1245,130.000000,-11,1
2,cooldown_long=30_cooldown_short=7,ETH,2326,2788,1391,1192,4180,64,1595.654917,8,-522,155.500000,81,1
3,cooldown_long=30_cooldown_short=7,SOL,2574,7719,455,282,8175,58,103.129434,13,-607,107.400000,225,1
4,cooldown_long=30_cooldown_short=7,XMR,2069,3586,418,324,4005,61,260.920101,4,-581,57.000000,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,cooldown_long=180_cooldown_short=60,LINK,2069,4373,853,1130,5226,74,9.143328,4,-2107,25.818182,191,100
15996,cooldown_long=180_cooldown_short=60,ETH,2069,3540,1494,721,5035,85,1769.763298,4,-1091,0.000000,118,100
15997,cooldown_long=180_cooldown_short=60,BTC,2069,4009,1974,1001,5984,94,6026.472653,4,-1905,155.000000,200,100
15998,cooldown_long=180_cooldown_short=60,ADA,2316,2307,916,1465,3223,60,0.432042,1,-2164,53.000000,19,100


In [7]:
search_res = history_search.groupby("strategy").agg(
    RoAC_mean=("RoAC", "mean"),
    RoAC_median=("RoAC", "median"),
    MDD_mean=("MDD", "mean"),
    MDD_median=("MDD", "median"),
    Portfolio_mean=("Portfolio", "mean"),
    Portfolio_median=("Portfolio", "median"),
    Cash_spent_mean=("Cash_spent", "mean"),
    Cash_spent_median=("Cash_spent", "median"),
    Total_value_mean=("Total_value", "mean"),
    Total_value_median=("Total_value", "median"),
    Total_value_low=("Total_value", lambda s: np.percentile(s, 25)),
    Total_value_high=("Total_value", lambda s: np.percentile(s, 75)),
)

search_res['Uncle'] = search_res['MDD_mean'] / search_res['Total_value_mean']
search_res['Profit_coef'] = search_res['Total_value_mean'] / search_res['Cash_spent_mean']
search_res.sort_values("RoAC_mean", ascending=False).style.background_gradient(cmap="RdYlGn", axis=0).format("{:.2f}")

,RoAC_mean,RoAC_median,MDD_mean,MDD_median,Portfolio_mean,Portfolio_median,Cash_spent_mean,Cash_spent_median,Total_value_mean,Total_value_median,Total_value_low,Total_value_high,Uncle,Profit_coef
strategy,,,,,,,,,,,,,,
cooldown_long=180_cooldown_short=60,90.42,46.50,-2966.03,-2197.00,1622.38,1344.00,2323.52,2316.00,6009.86,4997.00,3468.50,7647.50,-0.49,2.59
cooldown_long=90_cooldown_short=60,85.55,50.00,-1831.47,-1554.50,1393.95,1069.00,2353.41,2326.00,5897.36,5051.00,3614.00,7584.00,-0.31,2.51
cooldown_long=180_cooldown_short=30,84.40,47.00,-2870.02,-2115.00,1661.58,1412.00,2331.53,2316.00,5853.22,4804.00,3269.25,7463.75,-0.49,2.51
cooldown_long=180_cooldown_short=7,80.60,49.00,-2705.88,-2110.00,1555.66,1287.50,2327.59,2316.00,5897.08,4972.50,3534.25,7642.00,-0.46,2.53
cooldown_long=90_cooldown_short=30,78.48,57.00,-1554.04,-1343.00,1264.48,1064.50,2351.71,2326.00,5444.72,4667.00,3627.50,7003.50,-0.29,2.32
cooldown_long=180_cooldown_short=14,78.27,42.50,-2678.93,-2068.50,1592.37,1289.50,2345.08,2326.00,5807.25,4822.50,3450.50,7345.75,-0.46,2.48
cooldown_long=60_cooldown_short=60,73.70,52.00,-1504.20,-1256.50,1270.74,991.00,2394.49,2326.00,5474.71,4928.50,3602.50,7065.00,-0.27,2.29
cooldown_long=90_cooldown_short=14,73.15,44.50,-1465.38,-1273.00,1221.77,1040.50,2377.15,2326.00,5310.40,4769.00,3579.50,6831.25,-0.28,2.23
cooldown_long=60_cooldown_short=30,73.03,57.00,-1238.51,-1063.00,1214.89,981.00,2420.60,2326.00,5330.45,4805.00,3734.00,6681.00,-0.23,2.20
